# Folder 03 / file 02 — deploy Adult dest version

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n03_prod_cutover/n02_deploy.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Pin the approved Adult dest version for serving / @champion (and @prod if no canary).


## 1 — Imports


In [ ]:
from mlflow.exceptions import MlflowException
from src.n03_prod_cutover.n01_approval import resolve_cutover_dest_version
from src.n00_shared.runtime import Settings, configure_mlflow, load_settings, mlflow_client, set_task_value
from src.n00_shared.serving import serving_config, upsert_endpoint


## 2 — `_alias_version`


In [ ]:
def _alias_version(client, name: str, alias: str):
    try:
        return str(client.get_model_version_by_alias(name, alias).version)
    except MlflowException:
        return None
    except Exception:
        return None


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/2


In [ ]:
configure_mlflow(settings)
client = mlflow_client()
dest = settings.dest_model_name
version = resolve_cutover_dest_version(settings)
previous = _alias_version(client, dest, "prod")
client.set_registered_model_alias(dest, "champion", version)
if previous:
    client.set_model_version_tag(dest, version, "previous_prod_version", previous)
    set_task_value("previous_prod_version", previous)
else:
    client.set_model_version_tag(dest, version, "previous_prod_version", "")
    set_task_value("previous_prod_version", "")
first_or_no_canary = (not previous) or (not settings.allow_canary)
if first_or_no_canary:
    client.set_registered_model_alias(dest, "prod", version)
    upsert_endpoint(
        settings.endpoint_name,
        serving_config(dest, version, previous_version=None, canary_percent=100),
    )
    print(f"deploy set @prod and serving 100% to v{version}")
else:
    upsert_endpoint(
        settings.endpoint_name,
        serving_config(
            dest,
            version,
            previous_version=previous,
            canary_percent=settings.canary_percent,
        ),
    )
    print(f"deploy left @prod on {previous}; serving canary {settings.canary_percent}% -> v{version}")


## 5 — `run()` step 2/2


In [ ]:
set_task_value("model_version", version)
